# Custom Language ID — Small ECAPA-TDNN

Train a 5-language classifier (hi/en/ta/bn/mr) on IndicVoices + Svarah.
Run this notebook on a cloud GPU (Colab, Kaggle, etc.).

In [ ]:
# Cell 1 — Install dependencies (run once)
!pip install -q torch torchaudio numpy scipy onnx onnxruntime onnxscript datasets soundfile

In [ ]:
# Cell 2 — Imports and constants
import math
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import scipy.signal
import onnxruntime

TARGET_LANGS = ["hi", "en", "ta", "bn", "mr"]
NUM_CLASSES = len(TARGET_LANGS)
N_MELS = 80
N_FFT = 512
HOP_LENGTH = 160
WIN_LENGTH = 400
TARGET_SR = 16000
C = 512  # ECAPA channel size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU — this will be slow")

In [ ]:
# Cell 3 — Kaggle checkpoint directory
# On Kaggle, /kaggle/working is writable; regular /tmp may reset across sessions
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints → {CHECKPOINT_DIR}")

In [ ]:
# Cell 4 — Telephony augmentation
rng = np.random.default_rng(42)

def _mu_law_compress(x):
    mu = 255.0
    return np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)

def _mu_law_expand(y):
    mu = 255.0
    return np.sign(y) * (1.0 / mu) * (np.exp(np.abs(y) * np.log1p(mu)) - 1.0)

def simulate_telephony(audio, sr=TARGET_SR):
    if sr != 8000:
        audio = scipy.signal.resample_poly(audio, 8000, sr)
    sos = scipy.signal.butter(4, [300, 3400], btype="band", fs=8000, output="sos")
    audio = scipy.signal.sosfilt(sos, audio)
    audio = _mu_law_compress(audio)
    audio = _mu_law_expand(audio)
    noise = rng.normal(0, 0.002, audio.shape)
    audio = audio + noise
    frame_len = int(0.020 * 8000)
    for start in range(0, len(audio) - frame_len + 1, frame_len):
        if rng.random() < 0.02:
            audio[start:start + frame_len] = 0.0
    return scipy.signal.resample_poly(audio, TARGET_SR, 8000)

def maybe_augment(audio, prob=0.6):
    return simulate_telephony(audio) if rng.random() < prob else audio

In [ ]:
# Cell 5 — Build dataset from IndicVoices + Svarah
SAMPLES_PER_LANG = 2000  # adjust based on time/accuracy needs

indic_configs = {"hi": "hindi", "ta": "tamil", "bn": "bengali", "mr": "marathi"}
all_samples = {lang: [] for lang in TARGET_LANGS}

def _extract_audio(row):
    audio = row.get("audio") or row.get("audio_filepath")
    if audio is None:
        return None
    arr = np.asarray(audio["array"], dtype=np.float32)
    if audio["sampling_rate"] != TARGET_SR:
        arr = scipy.signal.resample_poly(arr, TARGET_SR, audio["sampling_rate"])
    if len(arr) < TARGET_SR // 2:
        return None
    return arr

for lang_code, config in indic_configs.items():
    stream = load_dataset("ai4bharat/IndicVoices", name=config, split="train", streaming=True)
    stream = stream.shuffle(buffer_size=256, seed=42)
    count = 0
    for row in stream:
        if count >= SAMPLES_PER_LANG:
            break
        audio = _extract_audio(row)
        if audio is not None:
            all_samples[lang_code].append(audio)
            count += 1
    print(f"{lang_code}: {count} samples")

stream = load_dataset("ai4bharat/Svarah", split="test", streaming=True)
stream = stream.shuffle(buffer_size=256, seed=42)
count = 0
for row in stream:
    if count >= SAMPLES_PER_LANG:
        break
    audio = _extract_audio(row)
    if audio is not None:
        all_samples["en"].append(audio)
        count += 1
print(f"en: {count} samples")

In [ ]:
# Cell 6 — Train/val/test split
rng_split = np.random.default_rng(42)
splits = {"train": {}, "val": {}, "test": {}}
for lang_code in TARGET_LANGS:
    audios = all_samples[lang_code]
    rng_split.shuffle(audios)
    n = len(audios)
    n_test = max(1, int(n * 0.1))
    n_val = max(1, int(n * 0.1))
    splits["test"][lang_code] = audios[:n_test]
    splits["val"][lang_code] = audios[n_test:n_test + n_val]
    splits["train"][lang_code] = audios[n_test + n_val:]

for split_name in ["train", "val", "test"]:
    counts = [len(splits[split_name][l]) for l in TARGET_LANGS]
    print(f"{split_name}: {counts} total={sum(counts)}")

In [ ]:
# Cell 7 — Mel-spectrogram transform + Dataset
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR, n_fft=N_FFT,
    win_length=WIN_LENGTH, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=0.0,
).to(device)

def audio_to_log_mel(audio_t):
    mel = mel_transform(audio_t)
    return torch.log(torch.clamp(mel, min=1e-10))

class LangIdDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.items = [(a, TARGET_LANGS.index(l)) for l in TARGET_LANGS for a in samples[l]]
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        audio, label = self.items[idx]
        if self.augment:
            audio = maybe_augment(audio)
        audio_t = torch.from_numpy(audio).float().to(device)
        return audio_to_log_mel(audio_t), torch.tensor(label, device=device)

train_ds = LangIdDataset(splits["train"], augment=True)
val_ds = LangIdDataset(splits["val"], augment=False)
test_ds = LangIdDataset(splits["test"], augment=False)

def collate_fn(batch):
    mels, labels = zip(*batch)
    max_len = max(m.shape[-1] for m in mels)
    padded = [F.pad(m, (0, max_len - m.shape[-1]), "constant", 0) for m in mels]
    lengths = torch.tensor([m.shape[-1] for m in mels])
    return torch.stack(padded), lengths, torch.stack(labels)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
# Cell 8 — ECAPA-TDNN model definition
class SEModule(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc1 = nn.Conv1d(channels, channels // reduction, kernel_size=1)
        self.fc2 = nn.Conv1d(channels // reduction, channels, kernel_size=1)
    def forward(self, x):
        w = F.adaptive_avg_pool1d(x, 1)
        w = F.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w))
        return x * w

class SE_Res2Block(nn.Module):
    def __init__(self, in_ch, out_ch, scale=8, dilation=1):
        super().__init__()
        self.scale = scale
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.convs = nn.ModuleList([
            nn.Conv1d(out_ch // scale, out_ch // scale, kernel_size=3,
                      padding=dilation, dilation=dilation,
                      groups=out_ch // scale)
            for _ in range(scale)
        ])
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.conv3 = nn.Conv1d(out_ch, out_ch, kernel_size=1)
        self.bn3 = nn.BatchNorm1d(out_ch)
        self.se = SEModule(out_ch)
        self.shortcut = nn.Identity() if in_ch == out_ch else nn.Conv1d(in_ch, out_ch, kernel_size=1)

    def forward(self, x):
        res = self.shortcut(x)
        x = F.relu(self.bn1(self.conv1(x)))
        sc = torch.chunk(x, self.scale, dim=1)
        out = []
        for i, conv in enumerate(self.convs):
            inp = sc[i] if i == 0 else sc[i] + out[i - 1]
            out.append(conv(inp))
        x = torch.cat(out, dim=1)
        x = self.bn2(x)
        x = self.bn3(self.conv3(x))
        x = self.se(x)
        return F.relu(x + res)

class AttentiveStatsPool(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.linear = nn.Conv1d(channels, channels, kernel_size=1)
    def forward(self, x):
        attn = torch.sigmoid(self.linear(x))
        mean = (attn * x).sum(dim=-1) / attn.sum(dim=-1).clamp(min=1e-10)
        std = (attn * (x - mean.unsqueeze(-1)) ** 2).sum(dim=-1).clamp(min=0).sqrt()
        return torch.cat([mean, std], dim=1)

class ECAPA_TDNN(nn.Module):
    def __init__(self, channels=C):
        super().__init__()
        self.conv1 = nn.Conv1d(N_MELS, channels, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(channels)
        self.block1 = SE_Res2Block(channels, channels, scale=8, dilation=2)
        self.block2 = SE_Res2Block(channels, channels, scale=8, dilation=3)
        self.block3 = SE_Res2Block(channels, channels, scale=8, dilation=4)
        self.pool = AttentiveStatsPool(channels)
        self.bn_pool = nn.BatchNorm1d(channels * 2)
        self.fc = nn.Linear(channels * 2, channels)
        self.bn_fc = nn.BatchNorm1d(channels)
        self.classifier = nn.Linear(channels, NUM_CLASSES)

    def forward(self, mel):
        x = F.relu(self.bn1(self.conv1(mel)))
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x)
        x = self.bn_pool(x)
        x = F.relu(self.bn_fc(self.fc(x)))
        return self.classifier(x)

model = ECAPA_TDNN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
# Cell 9 — Training loop
NUM_EPOCHS = 25
warmup_epochs = 5
steps_per_epoch = len(train_loader)
total_steps = NUM_EPOCHS * steps_per_epoch

def lr_lambda(step):
    epoch = step / max(steps_per_epoch, 1)
    if epoch < warmup_epochs:
        return epoch / warmup_epochs
    progress = (epoch - warmup_epochs) / max(NUM_EPOCHS - warmup_epochs, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
best_val_acc = 0.0
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    t0 = time.time()
    total_loss = 0.0

    for mels, lengths, labels in train_loader:
        optimizer.zero_grad()
        logits = model(mels)
        loss = F.cross_entropy(logits, labels, label_smoothing=0.1)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]

    # Validation
    model.eval()
    correct, total = 0, 0
    per_lang = {l: {"c": 0, "t": 0} for l in TARGET_LANGS}
    with torch.no_grad():
        for mels, lengths, labels in val_loader:
            preds = model(mels).argmax(dim=-1)
            for i in range(len(labels)):
                lang = TARGET_LANGS[labels[i].item()]
                per_lang[lang]["t"] += 1
                if preds[i] == labels[i]:
                    correct += 1
                    per_lang[lang]["c"] += 1
                total += 1

    val_acc = correct / max(total, 1) * 100
    per_lang_str = " | ".join(
        f"{l}: {d['c']/max(d['t'],1)*100:.1f}%"
        for l, d in per_lang.items()
    )
    print(f"E{epoch:3d} loss={total_loss/len(train_loader):.4f} lr={lr_now:.1e} "
          f"val={val_acc:.2f}% | {per_lang_str} | {time.time()-t0:.0f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), CHECKPOINT_DIR / "best_model.pt")
        print(f"  → saved ({val_acc:.2f}%)")
    else:
        patience_counter += 1
        if patience_counter >= 7:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val: {best_val_acc:.2f}%")

In [ ]:
# Cell 9b — Continued training from checkpoint (run after Cell 9)
# Picks up best_model.pt and runs 20 more epochs with a fresh cosine schedule
CONTINUE_EPOCHS = 20
state = torch.load(CHECKPOINT_DIR / "best_model.pt", map_location=DEVICE)
model.load_state_dict(state)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

def cont_lr_lambda(epoch):
    """Cosine from 5e-5 down to 1e-6 over CONTINUE_EPOCHS, no warmup."""
    return 0.5 * (1.0 + math.cos(math.pi * epoch / max(CONTINUE_EPOCHS, 1)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, cont_lr_lambda)
best_val_acc = 0.0
patience_counter = 0

for epoch in range(CONTINUE_EPOCHS):
    model.train()
    t0 = time.time()
    total_loss = 0.0

    for mels, lengths, labels in train_loader:
        optimizer.zero_grad()
        logits = model(mels)
        loss = F.cross_entropy(logits, labels, label_smoothing=0.1)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]

    model.eval()
    correct, total = 0, 0
    per_lang = {l: {"c": 0, "t": 0} for l in TARGET_LANGS}
    with torch.no_grad():
        for mels, lengths, labels in val_loader:
            preds = model(mels).argmax(dim=-1)
            for i in range(len(labels)):
                lang = TARGET_LANGS[labels[i].item()]
                per_lang[lang]["t"] += 1
                if preds[i] == labels[i]:
                    correct += 1
                    per_lang[lang]["c"] += 1
                total += 1

    val_acc = correct / max(total, 1) * 100
    per_lang_str = " | ".join(
        f"{l}: {d['c']/max(d['t'],1)*100:.1f}%"
        for l, d in per_lang.items()
    )
    print(f"Cont{epoch:>2d}/{CONTINUE_EPOCHS} loss={total_loss/len(train_loader):.4f} lr={lr_now:.1e} "
          f"val={val_acc:.2f}% | {per_lang_str} | {time.time()-t0:.0f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), CHECKPOINT_DIR / "best_model.pt")
        print(f"  → saved ({val_acc:.2f}%)")
    else:
        patience_counter += 1
        if patience_counter >= 7:
            print(f"Early stopping at continue epoch {epoch}")
            break

print(f"\nBest val after continued training: {best_val_acc:.2f}%")

In [ ]:
# Cell 10 — Test evaluation
state = torch.load(CHECKPOINT_DIR / "best_model.pt", map_location=DEVICE)
model.load_state_dict(state)
model.eval()

correct, total = 0, 0
per_lang = {l: {"c": 0, "t": 0} for l in TARGET_LANGS}
latencies = []

with torch.no_grad():
    for mels, lengths, labels in test_loader:
        t0 = time.perf_counter()
        preds = model(mels).argmax(dim=-1)
        latencies.append((time.perf_counter() - t0) * 1000)
        for i in range(len(labels)):
            lang = TARGET_LANGS[labels[i].item()]
            per_lang[lang]["t"] += 1
            if preds[i] == labels[i]:
                correct += 1
                per_lang[lang]["c"] += 1
            total += 1

latencies.sort()
print(f"\n{'Lang':<6} {'Acc':<8}")
print("-" * 14)
for l in TARGET_LANGS:
    d = per_lang[l]
    acc = d["c"] / max(d["t"], 1) * 100
    print(f"{l:<6} {acc:<7.1f}%")
total_acc = correct / max(total, 1) * 100
print(f"\nTest acc: {total_acc:.2f}%")
print(f"p50 latency: {latencies[len(latencies)//2]:.1f}ms")
print(f"p95 latency: {latencies[int(len(latencies)*0.95)]:.1f}ms")

In [ ]:
# Cell 11 — Export to ONNX
model.cpu()
model.eval()
dummy = torch.randn(1, N_MELS, 100)

torch.onnx.export(
    model, dummy, f"{CHECKPOINT_DIR}/lang_id_ecapa_ta.onnx",
    input_names=["mel"],
    output_names=["logits"],
    dynamic_axes={"mel": {0: "batch", 2: "time"}, "logits": {0: "batch"}},
    opset_version=18,
    dynamo=False,
)

# Validate
session = onnxruntime.InferenceSession(f"{CHECKPOINT_DIR}/lang_id_ecapa_ta.onnx")
ort_out = session.run(None, {"mel": dummy.numpy()})[0]
pt_out = model(dummy).detach().numpy()
diff = np.max(np.abs(ort_out - pt_out))
print(f"ONNX validated — max diff: {diff:.2e}")